# Aprendizado de Máquina — Lista prática 09

## Métricas para Classificação

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

O `bank_train_redux.csv` tem 200 variáveis anônimas e cerca de 10% de clientes
positivos. É o cenário em que a acurácia para de funcionar — e onde toda decisão
sobre o **corte** vale mais do que qualquer troca de modelo.

> **um classificador pode ordenar perfeitamente e ainda assim classificar
> *tudo* como negativo. Ordenar e decidir são duas etapas separadas.**

O último exercício investiga uma afirmação da nota de aula que **não se reproduz**
nestes dados — e descobre por quê.

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import os
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

import sklearn.model_selection as skm
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             confusion_matrix, precision_recall_curve,
                             roc_auc_score, roc_curve)

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — a acurácia que não diz nada

Carregue o banco. A última coluna do arquivo veio suja da exportação
(`var_199;;;;;;;`), e há `;` no meio de alguns números — a limpeza já está
escrita, mas vale ler: pré-processamento de dado real é assim.

In [ ]:
_nome = "bank_train_redux.csv"

# procura em dois lugares, sem baixar nada da internet: a pasta deste
# notebook primeiro ou então ../../recursos/dados/
_lugares = [_nome, os.path.join("..", "..", "recursos", "dados", _nome)]
_caminho = next((c for c in _lugares if os.path.exists(c)), None)

if _caminho is None:
    raise FileNotFoundError(
        f"nao encontrei '{_nome}'. Procurei nesta pasta e em "
        "../../recursos/dados/. Ponha o .csv ao lado deste notebook, "
        "ou mude o caminho se for necessário."
    )

banco = pd.read_csv(_caminho, nrows=40_000)
banco = banco.replace(to_replace=";", value="", regex=True)
banco = banco.rename(columns={"var_199;;;;;;;": "var_199"})
banco["var_199"] = pd.to_numeric(banco["var_199"])

Xb = banco.drop(columns=["ID_code", "target"]).astype(float).values
yb = banco["target"].values

X_tr, X_te, y_tr, y_te = skm.train_test_split(
    Xb, yb, test_size=0.3, random_state=2026, stratify=yb)      # (a) preserve a proporcao

print(f"n = {Xb.shape[0]}, d = {Xb.shape[1]}, prevalencia = {yb.mean():.4f}")
print(f"teste: {len(y_te)} clientes, {int(y_te.sum())} positivos")

In [ ]:
logistica = Pipeline([("escala", StandardScaler()),
                      ("modelo", LogisticRegression(max_iter=2000))]).fit(X_tr, y_tr)

p = logistica.predict_proba(X_te)[:, 1]                         # (a) a coluna da classe positiva

print(f"acuracia do modelo (corte 0,5): {np.mean((p > 0.5).astype(int) == y_te):.4f}")
print(f"acuracia do trivial (sempre 0): {np.mean(y_te == 0):.4f}")   # (b)
print(f"quantos o modelo chama de positivo: {int((p > 0.5).sum())} de {len(y_te)}")

Deve imprimir `n = 40000, d = 200, prevalencia = 0.0985`, `teste: 12000
clientes, 1182 positivos`, e depois:

```
acuracia do modelo (corte 0,5): 0.9155
acuracia do trivial (sempre 0): 0.9015
quantos o modelo chama de positivo: 484 de 12000
```

O modelo ganha **1,4 ponto percentual** de um programa que devolve `0` sempre. E
a terceira linha mostra por quê: das 12 000 observações, ele só se dispõe a
chamar 484 de positivas — menos da metade dos 1 182 positivos que existem.

Não é que o modelo seja ruim (o Exercício 3 mostra que ele ordena bem). É que o
corte $0{,}5$ está errado para este problema: com prevalência de 10%,
pouquíssimas observações chegam a $p > 0{,}5$.

---
## Exercício 2 — a matriz de confusão e o corte por custo

Comece pelo corte padrão, depois use a fórmula do Exercício 2 da Lista Teórica 09:

$$p^\ast = \frac{c_{FP}}{c_{FP}+c_{FN}}.$$

In [ ]:
def relatorio(y_real, prob, corte, rotulo):
    vn, fp, fn, vp = confusion_matrix(y_real, (prob > corte).astype(int)).ravel()   # (a)
    precisao = vp / (vp + fp)
    recall = vp / (vp + fn)                                     # (b)
    f1 = 2 * vp / (2 * vp + fp + fn)
    print(f"{rotulo:26s} corte {corte:.4f}  VP {vp:4d}  FP {fp:4d}  FN {fn:4d}"
          f"   precisao {precisao:.4f}  recall {recall:.4f}  F1 {f1:.4f}")
    return fp, fn


fp0, fn0 = relatorio(y_te, p, 0.5, "corte padrao")

In [ ]:
for c_fp, c_fn in [(1, 10), (1, 50)]:
    corte = c_fp / (c_fp + c_fn)                                # (a)
    fp1, fn1 = relatorio(y_te, p, corte, f"c_FP={c_fp}, c_FN={c_fn}")
    custo_otimo = c_fp * fp1 + c_fn * fn1                       # (b)
    custo_padrao = c_fp * fp0 + c_fn * fn0
    print(f"{'':26s} custo {custo_otimo:6d}  contra {custo_padrao:6d} no corte 0,5"
          f"   ({100 * (custo_otimo / custo_padrao - 1):+.0f}%)")

Deve imprimir:

```
corte padrao               corte 0.5000  VP  326  FP  158  FN  856   precisao 0.6736  recall 0.2758  F1 0.3914
c_FP=1, c_FN=10            corte 0.0909  VP  919  FP 2542  FN  263   precisao 0.2655  recall 0.7775  F1 0.3959
                           custo   5172  contra   8718 no corte 0,5   (-41%)
c_FP=1, c_FN=50            corte 0.0196  VP 1136  FP 6751  FN   46   precisao 0.1440  recall 0.9611  F1 0.2505
                           custo   9051  contra  42958 no corte 0,5   (-79%)
```

Três leituras.

**O corte é a decisão, não o modelo.** É o *mesmo* vetor `p` nas três linhas. Só
mudando onde se corta, o *recall* vai de 0,28 a 0,96 e a precisão de 0,67 a 0,14.
Trocar de modelo raramente produz um efeito dessa magnitude.

**A economia é grande.** Com $c_{FN}=50$, mover o corte de $0{,}5$ para $0{,}02$
corta o custo em **79%**, de 42 958 para 9 051 — sem ajustar nada.

**O $F_1$ não serve como critério aqui.** Ele é máximo em $0{,}3959$ no corte
$0{,}0909$, mas cai para $0{,}2505$ no corte que minimiza o custo com
$c_{FN}=50$. O $F_1$ trata precisão e *recall* como igualmente importantes, e
essa é justamente a suposição que o enunciado do problema contradiz. Quando os
custos são conhecidos, **otimize o custo**, não uma métrica genérica.

---
## Exercício 3 — as curvas que não dependem do corte

O Exercício 2 mostrou que o corte muda tudo. As curvas ROC e precisão--*recall*
resolvem isso avaliando **todos** os cortes de uma vez.

In [ ]:
auc = roc_auc_score(y_te, p)                                    # (a)
ap = average_precision_score(y_te, p)                           # (b)

print(f"AUC                 = {auc:.4f}   (acaso = 0,5)")
print(f"average precision   = {ap:.4f}   (acaso = prevalencia = {y_te.mean():.4f})")

In [ ]:
fpr, tpr, _ = roc_curve(y_te, p)
precisao, recall, _ = precision_recall_curve(y_te, p)

fig, (ax1, ax2) = subplots(1, 2, figsize=(9, 3.4))

ax1.plot(fpr, tpr, lw=1.6)
ax1.plot([0, 1], [0, 1], ls="--", lw=1, color="gray")
ax1.set_xlabel("taxa de falsos positivos")
ax1.set_ylabel("recall (taxa de verdadeiros positivos)")
ax1.set_title(f"ROC — AUC = {auc:.4f}")

ax2.plot(recall, precisao, lw=1.6)
ax2.axhline(y_te.mean(), ls="--", lw=1, color="gray")           # (a) a linha de base da PR
ax2.set_xlabel("recall")
ax2.set_ylabel("precisao")
ax2.set_title(f"precisao-recall — AP = {ap:.4f}")

fig.tight_layout()

Deve imprimir `AUC = 0.8529` e `average precision = 0.4933`, com prevalência
$0{,}0985$.

Os dois números contam histórias diferentes sobre o mesmo modelo, e as duas são
verdadeiras.

A **AUC de 0,85** diz que, sorteando um positivo e um negativo ao acaso, o modelo
dá escore maior ao positivo em 85% das vezes. É um bom desempenho de ordenação —
e confirma que o problema do Exercício 1 era o corte, não o modelo.

A **AP de 0,49** parece pior, mas a comparação certa é com a linha de base: um
classificador ao acaso teria AP igual à prevalência, $0{,}0985$. O modelo está
**cinco vezes** acima disso.

A diferença entre as duas é que a ROC usa a taxa de falsos positivos, cujo
denominador são os $10\,818$ negativos — um número grande, que dilui os falsos
positivos. A curva PR usa a precisão, cujo denominador são só os casos
classificados como positivos. **Em problema desbalanceado, a curva PR é a mais
informativa**, porque ela mede o que o usuário do sistema de fato experimenta:
de cada alerta emitido, quantos valem a pena.

---
## Exercício 4 — ordenar bem não é estimar bem (às vezes)

A nota de aula afirma que o naive Bayes empata com a logística em AUC e perde
feio em Brier, porque multiplica evidências correlacionadas como se fossem
independentes. Teste a afirmação **neste banco**.

In [ ]:
naive = GaussianNB().fit(X_tr, y_tr)
q = naive.predict_proba(X_te)[:, 1]

for nome, v in (("logistica", p), ("naive Bayes", q)):
    print(f"{nome:12s} AUC {roc_auc_score(y_te, v):.4f}   "
          f"Brier {brier_score_loss(y_te, v):.4f}   "                 # (a)
          f"fracao com p>0,9: {np.mean(v > 0.9):.4f}")
print(f"{'prevalencia':12s} {y_te.mean():.4f}")

Deve imprimir:

```
logistica    AUC 0.8529   Brier 0.0664   fracao com p>0,9: 0.0024
naive Bayes  AUC 0.8868   Brier 0.0600   fracao com p>0,9: 0.0071
prevalencia  0.0985
```

**A afirmação não se reproduz.** O naive Bayes não empata em AUC — ele *ganha*
(0,8868 contra 0,8529) — e o Brier dele é *melhor*, não pior.

Antes de concluir que a nota está errada, vale investigar a hipótese que ela
usa. O próximo passo faz isso.

A explicação do naive Bayes mal calibrado depende de **as covariáveis serem
correlacionadas**. Meça a correlação neste banco.

In [ ]:
matriz = np.corrcoef(X_tr.T)                                    # (a) correlacao entre COLUNAS
fora_da_diagonal = matriz[~np.eye(matriz.shape[0], dtype=bool)]

print(f"correlacao absoluta media entre as 200 variaveis: {np.abs(fora_da_diagonal).mean():.4f}")   # (b)
print(f"maxima: {np.abs(fora_da_diagonal).max():.4f}")

Deve imprimir correlação média **0,0048** e máxima **0,0266**.

Aí está a resposta: neste banco as 200 variáveis são **praticamente
independentes**. A suposição do naive Bayes, que costuma ser grosseiramente falsa,
aqui é quase exata — e um modelo cuja suposição é verdadeira e que estima só
$4d+1 = 801$ parâmetros (contra os 201 da logística, mas sem precisar de
otimização iterativa) tem tudo para ir bem.

A nota não está errada; ela está **condicionada** a uma hipótese que este conjunto
de dados não satisfaz. O exercício seguinte isola essa hipótese.

Para separar causa de coincidência, construa uma população em que você controla a
correlação. Seis covariáveis, 8% de positivos, e um único parâmetro $\rho$
regulando a dependência entre elas.

In [ ]:
def cenario(n, rng, prev=0.08, rho=0.85, d=6):
    y = (rng.uniform(size=n) < prev).astype(int)
    S = np.full((d, d), rho) + np.eye(d) * (1 - rho)
    L = np.linalg.cholesky(S)                       # covariancia S = L L'
    X = rng.normal(size=(n, d)) @ L.T
    X[y == 1] += 1.05                               # as classes diferem so na media
    return X, y


for rho in (0.85, 0.0):                                        # (a) e (b)
    X_s, y_s = cenario(4000, np.random.default_rng(2026), rho=rho)
    X_st, y_st = cenario(20000, np.random.default_rng(44), rho=rho)

    print(f"=== rho = {rho} ===")
    for nome, modelo in [("logistica", Pipeline([("e", StandardScaler()),
                                                 ("m", LogisticRegression(max_iter=2000))])),
                         ("naive Bayes", GaussianNB())]:
        v = modelo.fit(X_s, y_s).predict_proba(X_st)[:, 1]
        print(f"  {nome:12s} AUC {roc_auc_score(y_st, v):.4f}   "
              f"Brier {brier_score_loss(y_st, v):.4f}   "
              f"fracao com p>0,9: {np.mean(v > 0.9):.4f}")
    print(f"  prevalencia real: {y_st.mean():.4f}")

Deve imprimir:

```
=== rho = 0.85 ===
  logistica    AUC 0.7715   Brier 0.0655   fracao com p>0,9: 0.0000
  naive Bayes  AUC 0.7761   Brier 0.1499   fracao com p>0,9: 0.1070
  prevalencia real: 0.0794
=== rho = 0.0 ===
  logistica    AUC 0.9618   Brier 0.0322   fracao com p>0,9: 0.0226
  naive Bayes  AUC 0.9619   Brier 0.0323   fracao com p>0,9: 0.0225
  prevalencia real: 0.0794
```

Agora está tudo no lugar, e a explicação da nota se confirma **exatamente**,
sob a condição que ela pressupõe.

Com $\rho=0{,}85$: os dois **empatam em AUC** (0,7761 contra 0,7715) e o Brier
do naive Bayes é **2,3 vezes pior** (0,1499 contra 0,0655). A última coluna
mostra o mecanismo: ele atribui $p>0{,}9$ a **10,7%** das observações, quando só
$7{,}9\%$ são positivas — mais gente ele declara quase certa do que existe de
positivo. É o excesso de confiança de contar seis evidências correlacionadas como
se fossem seis evidências independentes.

Com $\rho=0$: as duas linhas são **indistinguíveis** até a terceira casa, em AUC
e em Brier. Sem correlação, não há evidência repetida a contar duas vezes, e o
naive Bayes é simplesmente um bom modelo.

Fechando o círculo: o banco tem correlação média $0{,}0048$, ou seja, está no
regime $\rho \approx 0$. Era de se esperar que o naive Bayes fosse bem ali — e
foi.

**A lição não é sobre naive Bayes.** É sobre como ler um aviso: *"o naive Bayes é
mal calibrado"* é a versão curta de *"o naive Bayes é mal calibrado quando as
covariáveis são correlacionadas"*, e a diferença entre as duas frases é a
diferença entre repetir uma regra e saber quando ela se aplica. Medir a
correlação custou uma linha.

> **Sua vez.** Rode o `cenario` com $\rho = 0{,}3$ e $\rho = 0{,}6$. A degradação
> do Brier do naive Bayes é gradual ou tem um limiar?

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | acurácia 0,9155 contra 0,9015 do classificador trivial — 1,4 ponto |
| 2 | só mudando o corte, o *recall* vai de 0,28 a 0,96 e o custo cai **79%** |
| 2 | o $F_1$ é máximo num corte que **não** é o de menor custo |
| 3 | AUC 0,8529 e AP 0,4933, contra uma linha de base de 0,0985 |
| 4 | neste banco o naive Bayes **ganha** da logística — o oposto do que a nota prevê |
| 4 | a correlação média entre as 200 variáveis é 0,0048: a hipótese dele é quase exata aqui |
| 4 | com $\rho=0{,}85$ a previsão da nota se confirma: Brier **2,3× pior**, e $p>0{,}9$ para 10,7% dos casos |

**A seguir.** A Aula 10 traz um classificador construído sobre uma ideia
geométrica diferente — a margem — e que, por não estimar probabilidades, deixa a
questão da calibração de fora por construção.